# Polymarket Copy-Trading Backtest — Colab Runner

**Click `Runtime → Run all`** (or Ctrl+F9 / ⌘+F9). Takes ~10 minutes.

What this does:
1. Clones the repo and installs deps
2. Downloads Jon-Becker's Polymarket trade dump (multi-GB; the slow step)
3. Converts it into our cache format
4. Runs the rolling walk-forward backtest twice:
   - **Recommended config** (pool=50 by lifetime PnL → top-15 by selection-window PnL)
   - **Honest baseline** (same but random pool — removes leaderboard look-ahead bias)
5. Prints both result tables for comparison

If both tables show positive mean return and Sharpe > 0.3, the strategy has real edge. If only the recommended config is positive, it's survivorship bias and won't generalize.

## Step 1 — Install + clone (~30 s)

In [ ]:
%%bash
set -e
apt-get -qq install -y zstd > /dev/null
if [ ! -d Crazy ]; then
  git clone -q --depth 1 -b claude/polymarket-copy-trading-sim-LR3Py https://github.com/calvinling2021-star/Crazy.git
fi
pip install -q -r Crazy/polymarket/requirements.txt
echo 'install OK'

## Step 2 — Download trade data (~5–10 min, multi-GB)

If this step hangs or 403s, scroll to **Step 2b** below for the Dune Analytics fallback.

In [ ]:
%%bash
set -e
cd Crazy
if [ ! -d data/polymarket ]; then
  echo 'Downloading data.tar.zst (this is the slow step)…'
  curl -L --fail --progress-bar -o data.tar.zst https://s3.jbecker.dev/data.tar.zst
  echo 'Extracting…'
  zstd -d data.tar.zst -c | tar -xf -
  rm data.tar.zst
fi
echo; echo 'data/polymarket/ contents:'
ls -lh data/polymarket/ | head -20

## Step 3 — Inspect parquet schema (~5 s)

Confirms which column names jbecker's current dump uses, so the converter knows where to find them.

In [ ]:
%cd Crazy
!python3 -m polymarket.sources.jbecker --data-dir data --inspect 2>&1 | head -60

## Step 4 — Convert to cache.json (~1–3 min depending on dataset size)

In [ ]:
!python3 -m polymarket.sources.jbecker --data-dir data --top 200 --window-months 12 --out cache.json --log-level INFO 2>&1 | tail -25

## Step 5 — Backtest: recommended config (pnl pool, top-15)

In [ ]:
!python3 -W ignore -m polymarket.backtest --cache cache.json --rolling \
  --rolling-windows 6 --rolling-step-months 1 \
  --rolling-sel-months 6 --rolling-val-months 1 \
  --candidate-pool 50 --top-k 15 \
  --min-consensus-leaders 2 --consensus-window-hours 24 \
  --fraction 0.02 --log-level WARNING --out results/recommended

## Step 6 — Backtest: honest baseline (random pool, removes look-ahead)

In [ ]:
!python3 -W ignore -m polymarket.backtest --cache cache.json --rolling \
  --rolling-windows 6 --rolling-step-months 1 \
  --rolling-sel-months 6 --rolling-val-months 1 \
  --candidate-pool 50 --top-k 15 --pool-rank-by random \
  --min-consensus-leaders 2 --consensus-window-hours 24 \
  --fraction 0.02 --log-level WARNING --out results/honest

## Step 7 — Verdict

Reads both summary.json files and prints a side-by-side comparison.

In [ ]:
import json, pathlib
from tabulate import tabulate
rows = []
for label, path in [('recommended (pnl pool)', 'results/recommended/summary.json'),
                    ('honest (random pool)',   'results/honest/summary.json')]:
    p = pathlib.Path(path)
    if not p.exists():
        rows.append([label, 'NO OUTPUT — backtest step failed'])
        continue
    s = json.loads(p.read_text())
    rows.append([
        label,
        f"mean ret/mo: {s.get('mean_return_pct', 0):+.2f}%\n"
        f"mean Sharpe: {s.get('mean_sharpe', 0):+.2f}\n"
        f"positive windows: {s.get('positive_windows', 0)}/{s.get('n_windows', 0)}\n"
        f"worst DD: {s.get('worst_max_dd_pct', 0):+.2f}%\n"
        f"wallet stability: {s.get('wallet_set_stability', 0):.2f}",
    ])
print(tabulate(rows, tablefmt='grid'))
print()
rec = json.loads(open('results/recommended/summary.json').read()) if pathlib.Path('results/recommended/summary.json').exists() else {}
hon = json.loads(open('results/honest/summary.json').read()) if pathlib.Path('results/honest/summary.json').exists() else {}
if rec and hon:
    rec_ret, hon_ret = rec.get('mean_return_pct', 0), hon.get('mean_return_pct', 0)
    rec_shp, hon_shp = rec.get('mean_sharpe', 0), hon.get('mean_sharpe', 0)
    if hon_ret > 0 and hon_shp > 0.3:
        print('VERDICT: honest baseline is positive — real edge worth investigating.')
    elif rec_ret > 0 and hon_ret <= 0:
        print('VERDICT: only the biased pool is positive — likely survivorship bias, won\'t generalize.')
    else:
        print('VERDICT: neither config shows clean edge — strategy needs more work or different signal.')